In [0]:
from datetime import datetime
from pyspark.sql import Row
from uuid import uuid4

def write_log(step, status, row_count, message):

    spark.createDataFrame([
        Row(run_id=str(uuid4()),
            run_timestamp=datetime.now(),
            step=step,
            status=status,
            row_count=row_count,
            message=message
        )
    ]).write \
      .mode("append") \
      .saveAsTable(
          "training.monitoring.pipeline_logs"
      )


print("=" * 50)
print("SILVER START")
print("=" * 50)

try:

    bronze_df = spark.table(
        "training.bronze.orders_raw"
    )

    bronze_count = bronze_df.count()

    print(f"Avant nettoyage : {bronze_count}")

    bronze_df.show()

    silver_df = (
        bronze_df
            .filter("amount IS NOT NULL")
            .filter("customer_id IS NOT NULL")
            .filter("amount > 0")
            .dropDuplicates(["order_id"])
        )

    silver_df = silver_df.select(
            "order_id",
            "customer_id",
            "country",
            "amount",
            "order_date")

    silver_count = silver_df.count()

    print("Après nettoyage")

    silver_df.show()

    silver_df.write \
        .format("delta") \
        .mode("overwrite") \
        .saveAsTable(
            "training.silver.orders_clean"
        )

    print(f"Bronze rows : {bronze_count}")
    print(f"Silver rows : {silver_count}")

    write_log(
        step="SILVER",
        status="SUCCESS",
        row_count=silver_count,
        message=f"Nettoyage terminé. Bronze={bronze_count}, Silver={silver_count}"
    )

except Exception as e:

    write_log(
        step="SILVER",
        status="FAILED",
        row_count=0,
        message=str(e)
    )

    raise

print("=" * 50)
print("SILVER END")
print("=" * 50)


